<a href="https://colab.research.google.com/github/pikachu669/MLtbbala/blob/master/Triauto_Color_Grade_Video_From_Photo_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Color Grade Video From Photo Reference
Extracts color, brightness, and tone from your reference photo and applies it to your video.

**Steps:** Run each cell top to bottom.

In [ ]:
# CELL 1 - Install Dependencies
import subprocess, sys

pkgs = ['opencv-python-headless', 'numpy', 'Pillow', 'tqdm', 'requests']
print('Installing dependencies...\n')
for p in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], capture_output=True)
    print(f'  OK  {p}' if r.returncode == 0 else f'  FAIL  {p}')

# CuPy for GPU-accelerated color grading (matches Colab's CUDA version)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'], capture_output=True)
print('  OK  cupy-cuda12x' if r.returncode == 0 else '  FAIL  cupy-cuda12x (GPU grading will be skipped)')

import cv2, numpy as np, requests, os, time, base64, subprocess
from fractions import Fraction
from tqdm.notebook import tqdm
from IPython.display import display, HTML
import warnings; warnings.filterwarnings('ignore')
print('\nAll dependencies ready!')
print('TIP: Runtime > Change Runtime Type > GPU for faster processing.')

Installing dependencies...

  OK  opencv-python-headless
  OK  numpy
  OK  Pillow
  OK  tqdm
  OK  requests
  OK  cupy-cuda12x

All dependencies ready!
TIP: Runtime > Change Runtime Type > GPU for faster processing.


In [ ]:
# CELL 2 - Set Your URLs and Settings

PHOTO_URL  = 'https://static-ca-cdn.eporner.com/gallery/6O/4Q/Bmb2ylU4Q6O/14847757-14847757_296x1000.jpg'   # <-- paste your photo URL
VIDEO_URL  = 'https://hl-temp.olamdrive.com/download.aspx?file=feE2DoX7sGY9AG%2BBFm%2F1V2RJevaHhA1qa5nolR6Nrnfij6Q9T4upp%2BtIGVh0%2Fvvw&expiry=vl2mZqR%2BJnQoMN9GuDIjSQ%3D%3D&mac=6d28b248098fb30c45dd18d420d31c53dbdc07fa189bbe35ea16f0c8d6dca233'   # <-- paste your video URL

STRENGTH   = 1.0   # 0.0 = original, 1.0 = full transfer, 0.5 = blended
BRIGHTNESS = 1.0   # 1.0 = no change, 1.2 = 20% brighter
SATURATION = 1.0   # 1.0 = no change, 1.3 = 30% more vivid
OUTPUT_CRF = 24    # 0 = lossless, 18 = near lossless, 28 = smaller file
OUTPUT_FILE= 'color_graded_output.mp4'

# Google Drive output folder (must already be mounted)
DRIVE_FOLDER = '/content/drive/MyDrive/ColorGraded'

print('Settings:')
print(f'  Strength    : {STRENGTH}')
print(f'  Brightness  : {BRIGHTNESS}x')
print(f'  Saturation  : {SATURATION}x')
print(f'  Quality CRF : {OUTPUT_CRF}')
print(f'  Output      : {OUTPUT_FILE}')
print(f'  Drive folder: {DRIVE_FOLDER}')

Settings:
  Strength    : 1.0
  Brightness  : 1.0x
  Saturation  : 1.0x
  Quality CRF : 24
  Output      : color_graded_output.mp4
  Drive folder: /content/drive/MyDrive/ColorGraded


In [ ]:
# CELL 3 - Download Photo and Video

def download_file(url, filename, label):
    print(f'Downloading {label}...')
    r = requests.get(url, stream=True, headers={'User-Agent': 'Mozilla/5.0'}, timeout=120)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(filename, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True,
            unit_divisor=1024, desc=f'  {label}', colour='green', ncols=80) as bar:
        for chunk in r.iter_content(chunk_size=262144):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    mb = os.path.getsize(filename) / 1048576
    print(f'  Saved: {filename} ({mb:.1f} MB)\n')

photo_ext = os.path.splitext(PHOTO_URL.split('?')[0])[-1] or '.jpg'
video_ext = os.path.splitext(VIDEO_URL.split('?')[0])[-1] or '.mp4'
PHOTO_FILE = f'reference_photo{photo_ext}'
VIDEO_FILE = f'input_video{video_ext}'

download_file(PHOTO_URL, PHOTO_FILE, 'Reference Photo')
download_file(VIDEO_URL, VIDEO_FILE, 'Input Video')

img_check = cv2.imread(PHOTO_FILE)
cap_check = cv2.VideoCapture(VIDEO_FILE)
vid_ok  = cap_check.isOpened()
vw      = int(cap_check.get(cv2.CAP_PROP_FRAME_WIDTH))
vh      = int(cap_check.get(cv2.CAP_PROP_FRAME_HEIGHT))
vtotal  = int(cap_check.get(cv2.CAP_PROP_FRAME_COUNT))
vcv_fps = cap_check.get(cv2.CAP_PROP_FPS)
cap_check.release()

if img_check is None: raise ValueError('Could not read photo. Check URL.')
if not vid_ok: raise ValueError('Could not open video. Check URL.')

# Get exact FPS fraction from ffprobe
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=r_frame_rate,nb_frames',
     '-of', 'default=noprint_wrappers=1', VIDEO_FILE],
    capture_output=True, text=True
)
fps_str, nb_frames_str = None, None
for line in probe.stdout.strip().splitlines():
    if line.startswith('r_frame_rate='): fps_str = line.split('=')[1].strip()
    if line.startswith('nb_frames='):    nb_frames_str = line.split('=')[1].strip()

FPS_FRAC   = fps_str if fps_str else str(Fraction(vcv_fps).limit_denominator(1001))
FPS_FLOAT  = float(Fraction(FPS_FRAC))
# prefer ffprobe frame count; fall back to cv2
TOTAL_FRAMES = int(nb_frames_str) if nb_frames_str and nb_frames_str.isdigit() else vtotal

ph, pw = img_check.shape[:2]
print(f'Photo  : {pw}x{ph}px')
print(f'Video  : {vw}x{vh}px')
print(f'FPS    : {FPS_FRAC} = {FPS_FLOAT:.6f}')
print(f'Frames : {TOTAL_FRAMES}  ({TOTAL_FRAMES/FPS_FLOAT:.2f}s)')
print('Both files ready!')

  Reference Photo:   0%|                            | 0.00/21.2k [00:00<?, ?B/s]

  Saved: reference_photo.jpg (0.0 MB)



  Input Video:   0%|                                | 0.00/3.95G [00:00<?, ?B/s]

  Saved: input_video.aspx (4045.6 MB)

Photo  : 296x197px
Video  : 1920x1080px
FPS    : 24000/1001 = 23.976024
Frames : 65530  (2733.15s)
Both files ready!


In [ ]:
# CELL 4 - Color Grade the Video
# Frames are piped directly into ffmpeg — no cv2.VideoWriter so FPS is never altered.

import time

def sep(title):
    print(f'\n{"-"*55}\n  {title}\n{"-"*55}')

# ── GPU detection ─────────────────────────────────────────────
USE_GPU_GRADE = False
try:
    import cupy as cp
    _ = cp.zeros((1,))  # test allocation
    USE_GPU_GRADE = True
    print('  CuPy GPU grading: ENABLED')
except Exception as e:
    print(f'  CuPy not available ({e}); falling back to CPU grading.')

# Check for NVENC support in ffmpeg
USE_NVENC = False
try:
    enc_check = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'],
                                capture_output=True, text=True, timeout=30)
    if 'h264_nvenc' in enc_check.stdout:
        # quick smoke test: try a tiny encode
        test = subprocess.run(
            ['ffmpeg', '-y', '-f', 'lavfi', '-i', 'color=c=black:s=64x64:d=0.1',
             '-c:v', 'h264_nvenc', '-f', 'null', '-'],
            capture_output=True, timeout=30)
        USE_NVENC = (test.returncode == 0)
    print(f'  NVENC GPU encoding: {"ENABLED" if USE_NVENC else "NOT AVAILABLE (using libx264)"}')
except Exception as e:
    print(f'  NVENC check failed ({e}); using libx264.')

# ── STEP 1: Analyze reference photo ──────────────────────────
sep('STEP 1/3 - Analyzing Reference Photo')
ref_bgr  = cv2.imread(PHOTO_FILE)
ref_lab  = cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2LAB).astype('float32')
ref_stats = {}
ch_names  = ['L (Lightness)', 'A (Green-Red)', 'B (Blue-Yellow)']
for i, name in enumerate(tqdm(ch_names, desc='  Channels', colour='cyan', ncols=80)):
    ch = ref_lab[:, :, i]
    ref_stats[i] = {'mean': ch.mean(), 'std': ch.std()}
    time.sleep(0.05)
for i, name in enumerate(ch_names):
    print(f'  {name}: mean={ref_stats[i]["mean"]:.2f}  std={ref_stats[i]["std"]:.2f}')
print('  Done.')

# ── STEP 2: Grade frames, pipe raw RGB to ffmpeg ─────────────
sep('STEP 2/3 - Color Grading + Encoding (single pass)')
print(f'  Input FPS   : {FPS_FRAC} ({FPS_FLOAT:.6f})')
print(f'  Output FPS  : same — {FPS_FRAC}')
print(f'  Resolution  : {vw}x{vh}')
print(f'  Total frames: {TOTAL_FRAMES}')
print()

def grade_frame_cpu(frame, strength, brightness, saturation):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB).astype('float32')
    out = lab.copy()
    for i in range(3):
        ch = lab[:, :, i]
        t  = (ch - ch.mean()) * (ref_stats[i]['std'] / (ch.std() + 1e-6)) + ref_stats[i]['mean']
        out[:, :, i] = ch * (1 - strength) + t * strength
    result = cv2.cvtColor(np.clip(out, 0, 255).astype('uint8'), cv2.COLOR_LAB2BGR)
    if brightness != 1.0:
        hsv = cv2.cvtColor(result, cv2.COLOR_BGR2HSV).astype('float32')
        hsv[:, :, 2] = np.clip(hsv[:, :, 2] * brightness, 0, 255)
        result = cv2.cvtColor(hsv.astype('uint8'), cv2.COLOR_HSV2BGR)
    if saturation != 1.0:
        hsv = cv2.cvtColor(result, cv2.COLOR_BGR2HSV).astype('float32')
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation, 0, 255)
        result = cv2.cvtColor(hsv.astype('uint8'), cv2.COLOR_HSV2BGR)
    return result

if USE_GPU_GRADE:
    ref_mean_gpu = cp.asarray([ref_stats[i]['mean'] for i in range(3)], dtype=cp.float32)
    ref_std_gpu  = cp.asarray([ref_stats[i]['std']  for i in range(3)], dtype=cp.float32)

    def grade_frame_gpu(frame, strength, brightness, saturation):
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        lab_gpu = cp.asarray(lab, dtype=cp.float32)
        out = lab_gpu.copy()
        for i in range(3):
            ch = lab_gpu[:, :, i]
            ch_mean = ch.mean()
            ch_std  = ch.std()
            t = (ch - ch_mean) * (ref_std_gpu[i] / (ch_std + 1e-6)) + ref_mean_gpu[i]
            out[:, :, i] = ch * (1 - strength) + t * strength
        out = cp.clip(out, 0, 255).astype(cp.uint8)
        result_lab = cp.asnumpy(out)
        result = cv2.cvtColor(result_lab, cv2.COLOR_LAB2BGR)
        if brightness != 1.0:
            hsv = cv2.cvtColor(result, cv2.COLOR_BGR2HSV).astype('float32')
            hsv[:, :, 2] = np.clip(hsv[:, :, 2] * brightness, 0, 255)
            result = cv2.cvtColor(hsv.astype('uint8'), cv2.COLOR_HSV2BGR)
        if saturation != 1.0:
            hsv = cv2.cvtColor(result, cv2.COLOR_BGR2HSV).astype('float32')
            hsv[:, :, 1] = np.clip(hsv[:, :, 1] * saturation, 0, 255)
            result = cv2.cvtColor(hsv.astype('uint8'), cv2.COLOR_HSV2BGR)
        return result

    grade_frame = grade_frame_gpu
else:
    grade_frame = grade_frame_cpu

# ffmpeg reads raw BGR24 frames from stdin at the exact FPS fraction
if USE_NVENC:
    vcodec_args = [
        '-c:v', 'h264_nvenc',
        '-preset', 'p4',          # balanced GPU preset
        '-rc', 'vbr',
        '-cq', str(OUTPUT_CRF),
        '-b:v', '0',
    ]
else:
    vcodec_args = [
        '-c:v', 'libx264',
        '-preset', 'fast',
        '-crf', str(OUTPUT_CRF),
    ]

ffmpeg_cmd = [
    'ffmpeg', '-y',
    '-f', 'rawvideo',
    '-vcodec', 'rawvideo',
    '-s', f'{vw}x{vh}',
    '-pix_fmt', 'bgr24',
    '-r', FPS_FRAC,          # exact fraction e.g. 24000/1001 or 60/1 or 30000/1001
    '-i', 'pipe:0',          # graded frames from stdin
    '-i', VIDEO_FILE,        # original file for audio
    '-map', '0:v:0',
    '-map', '1:a?',
    *vcodec_args,
    '-c:a', 'aac',
    '-b:a', '192k',
    '-movflags', '+faststart',
    '-progress', 'pipe:2',   # encode progress on stderr
    '-nostats',
    OUTPUT_FILE
]

import threading

cap  = cv2.VideoCapture(VIDEO_FILE)
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE,
                        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)

t0   = time.time()
done = 0
pbar = tqdm(total=TOTAL_FRAMES, desc='  Grading+Encoding', unit='frame',
            colour='yellow', ncols=80)
enc_pbar = tqdm(total=TOTAL_FRAMES, desc='  Muxing audio', unit='frame',
                colour='magenta', ncols=80)

# IMPORTANT: ffmpeg's stderr (progress) pipe must be drained continuously,
# otherwise its buffer fills up and ffmpeg blocks -> stdin write also blocks -> hang.
stderr_lines = []
def drain_stderr():
    cf = 0
    for line in proc.stderr:
        line = line.decode('utf-8', errors='ignore').strip()
        stderr_lines.append(line)
        if line.startswith('frame='):
            try:
                nf = int(line.split('=')[1])
                enc_pbar.update(nf - cf); cf = nf
            except: pass

stderr_thread = threading.Thread(target=drain_stderr, daemon=True)
stderr_thread.start()

try:
    while True:
        ret, frame = cap.read()
        if not ret: break
        graded = grade_frame(frame, STRENGTH, BRIGHTNESS, SATURATION)
        proc.stdin.write(graded.tobytes())  # pipe raw bytes directly to ffmpeg
        done += 1
        pbar.update(1)
        if done % 60 == 0:
            el = time.time() - t0
            pbar.set_postfix({'fps': f'{done/el:.1f}', 'ETA': f'{(TOTAL_FRAMES-done)/(done/el):.0f}s'})
except BrokenPipeError:
    print('  ffmpeg pipe closed early — check stderr below.')
finally:
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass

pbar.close()

# ── STEP 3: Wait for ffmpeg to finish ────────────────────────
sep('STEP 3/3 - Finalizing Output')
proc.wait()
stderr_thread.join()
enc_pbar.close()

if proc.returncode != 0:
    print('  ffmpeg exited with error code', proc.returncode)
    print('  Last stderr lines:')
    for l in stderr_lines[-20:]:
        print('   ', l)

el_total = time.time() - t0

# Verify output FPS with ffprobe
v_probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
     '-show_entries', 'stream=r_frame_rate,nb_frames',
     '-of', 'default=noprint_wrappers=1', OUTPUT_FILE],
    capture_output=True, text=True
)
out_fps, out_frames = 'unknown', 'unknown'
for line in v_probe.stdout.strip().splitlines():
    if line.startswith('r_frame_rate='): out_fps    = line.split('=')[1].strip()
    if line.startswith('nb_frames='):    out_frames = line.split('=')[1].strip()

sz = os.path.getsize(OUTPUT_FILE) / 1048576 if os.path.exists(OUTPUT_FILE) else 0

print()
print('=' * 55)
print('  COLOR GRADING COMPLETE')
print('=' * 55)
print(f'  Input  FPS    : {FPS_FRAC} ({FPS_FLOAT:.6f})')
print(f'  Output FPS    : {out_fps} ({float(Fraction(out_fps)):.6f})')
print(f'  Input  frames : {TOTAL_FRAMES}')
print(f'  Output frames : {out_frames}')
match = 'MATCH' if str(out_frames) == str(TOTAL_FRAMES) else 'MISMATCH - check video'
print(f'  Frame check   : {match}')
print(f'  Output size   : {sz:.1f} MB')
print(f'  Total time    : {el_total:.1f}s')
print('=' * 55)
print('  Run Cell 5 to preview and upload to Drive.')

  CuPy GPU grading: ENABLED
  NVENC GPU encoding: NOT AVAILABLE (using libx264)

-------------------------------------------------------
  STEP 1/3 - Analyzing Reference Photo
-------------------------------------------------------


  Channels:   0%|                                         | 0/3 [00:00<?, ?it/s]

  L (Lightness): mean=137.47  std=51.10
  A (Green-Red): mean=138.49  std=17.90
  B (Blue-Yellow): mean=136.67  std=22.15
  Done.

-------------------------------------------------------
  STEP 2/3 - Color Grading + Encoding (single pass)
-------------------------------------------------------
  Input FPS   : 24000/1001 (23.976024)
  Output FPS  : same — 24000/1001
  Resolution  : 1920x1080
  Total frames: 65530



  Grading+Encoding:   0%|                          | 0/65530 [00:00<?, ?frame/s]

  Muxing audio:   0%|                              | 0/65530 [00:00<?, ?frame/s]

In [ ]:
# CELL 5 - Preview Before/After and Upload to Google Drive

# Before/After preview
cap_o = cv2.VideoCapture(VIDEO_FILE)
cap_g = cv2.VideoCapture(OUTPUT_FILE)
r1, fo = cap_o.read()
r2, fg = cap_g.read()
cap_o.release(); cap_g.release()

if r1 and r2:
    dw  = 460
    dh  = int(fo.shape[0] * dw / fo.shape[1])
    to  = cv2.resize(fo, (dw, dh))
    tg  = cv2.resize(fg, (dw, dh))
    _, b1 = cv2.imencode('.jpg', to)
    _, b2 = cv2.imencode('.jpg', tg)
    i1 = base64.b64encode(b1).decode()
    i2 = base64.b64encode(b2).decode()
    display(HTML(f'''
    <div style="font-family:sans-serif;padding:14px;background:#111;border-radius:10px;color:#fff;">
      <h3 style="color:#fff;margin-bottom:14px;">Before vs After (First Frame)</h3>
      <div style="display:flex;gap:14px;">
        <div style="text-align:center;">
          <p style="color:#aaa;margin-bottom:6px;font-size:13px;">ORIGINAL</p>
          <img src="data:image/jpeg;base64,{i1}" style="border-radius:6px;border:2px solid #444;"/>
        </div>
        <div style="text-align:center;">
          <p style="color:#f9c74f;margin-bottom:6px;font-size:13px;">COLOR GRADED</p>
          <img src="data:image/jpeg;base64,{i2}" style="border-radius:6px;border:2px solid #f9c74f;"/>
        </div>
      </div>
    </div>'''))

# Upload to Google Drive
print('\nUploading to Google Drive...')
if not os.path.exists('/content/drive/MyDrive'):
    print('  Drive not mounted. Mount it first: from google.colab import drive; drive.mount("/content/drive")')
else:
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    dest      = os.path.join(DRIVE_FOLDER, OUTPUT_FILE)
    sz_bytes  = os.path.getsize(OUTPUT_FILE)
    chunk     = 4 * 1024 * 1024
    up_pbar   = tqdm(total=sz_bytes, unit='B', unit_scale=True, unit_divisor=1024,
                     desc='  Uploading', colour='green', ncols=80)
    with open(OUTPUT_FILE, 'rb') as src_f, open(dest, 'wb') as dst_f:
        while True:
            buf = src_f.read(chunk)
            if not buf: break
            dst_f.write(buf)
            up_pbar.update(len(buf))
    up_pbar.close()
    if os.path.exists(dest):
        print(f'  Saved to: {dest}')
        print(f'  Size    : {os.path.getsize(dest)/1048576:.1f} MB')
        print('  Upload complete!')
    else:
        print('  Upload failed. Check Drive folder path or permissions.')

In [ ]:
from google.colab import drive; drive.mount("/content/drive")

Mounted at /content/drive
